In [40]:
#!unzip PySESM.zip
# !pip install torchmetrics

In [51]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import multivariate_normal
from torchmetrics import MeanSquaredError, Accuracy, Precision, Recall, F1Score
from sklearn.model_selection import RandomizedSearchCV

from PySESM.models.SESM.SESM import SESM_Model
from PySESM.base_functions.Function import GaussianFunctions


## Definicion de covarianzas no diagnonales

In [42]:

# Non-diagonal covariance
def generate_sigma_tensors():
    e0 = torch.tensor([1.0, 1.0], dtype=torch.float32)
    e0 = e0 / e0.norm()

    def generate_sigma(rotation_angle, scaling_factors):
        rotation_matrix = torch.tensor([[np.cos(rotation_angle), -np.sin(rotation_angle)],
                                       [np.sin(rotation_angle), np.cos(rotation_angle)]], dtype=torch.float32)
        e = torch.mm(rotation_matrix, e0.unsqueeze(1))
        E = torch.cat((e0.unsqueeze(1), e), dim=1)
        D = torch.diag(torch.tensor(scaling_factors, dtype=torch.float32))
        return torch.mm(torch.mm(E, D), E.t())

    sigma1 = generate_sigma(np.pi/4, [0.4, 0.1])
    sigma2 = generate_sigma(np.pi/4, [0.05, 0.5])
    sigma3 = generate_sigma(np.pi/4, [0.2, 0.5])

    return sigma1, sigma2, sigma3


In [43]:
sigma1, sigma2, sigma3 = generate_sigma_tensors()

## Definicion de varianzas diagonales

In [44]:
sigma1_d = 0.15 * torch.eye(2)
sigma2_d = 0.2 * torch.eye(2)
sigma3_d = 0.3 *torch.eye(2)

## Crear Gaussianas

In [45]:

def generate_mesh_and_z(sigma1, sigma2, sigma3):
    N_points = 50
    xl = -2
    xr = 2

    x = np.linspace(xl, xr, N_points+1)
    xx, yy = np.meshgrid(x, x)

    X = torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]), dtype=torch.float32)

    mu1 = torch.tensor([1, 1], dtype=torch.float32)
    mu2 = torch.tensor([1, -1], dtype=torch.float32)
    mu3 = torch.tensor([-1, -1], dtype=torch.float32)

    z1 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu1.numpy(), sigma1.numpy()), dtype=torch.float32)
    z2 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu2.numpy(), sigma2.numpy()), dtype=torch.float32)
    z3 = torch.tensor(multivariate_normal.pdf(X.numpy(), mu3.numpy(), sigma3.numpy()), dtype=torch.float32)

    zz = (z1 + z2 + z3).reshape(xx.shape)

    return xx, yy, zz




## Covarianza de gaussianas del experimento:
  - 3 Diagonales
  - 2 Diagonales, 1 no diagonal
  - 1 diagonal, 2 no diagonales
  - 3 no diagonales

In [46]:

def run_experiment(x, y, z, m_epochs, dict_epochs, h_epochs,expe, iter, debug=True):
  x_values = xx.ravel()
  y_values = yy.ravel()
  z_values = zz.ravel()

  n_samples = 50
  n_features = 2
  l_functions =  20
  total_points = len(x_values)

  min_separation = 20

  selected_indexes = []

  while len(selected_indexes) < n_samples:

      random_index = np.random.randint(total_points)

      if all(abs(random_index - existing_index) >= min_separation for existing_index in selected_indexes):
          selected_indexes.append(random_index)

  sampled_indices = selected_indexes

  sampled_x = torch.tensor(x_values[sampled_indices], dtype=torch.float32)
  sampled_y = torch.tensor(y_values[sampled_indices], dtype=torch.float32)

  X = torch.stack((sampled_x, sampled_y), dim=1)
  y = z_values[sampled_indices].clone().detach().to(dtype=torch.float32)

  gaussian_function = GaussianFunctions(n_features= n_features, n_functions = l_functions)
  model = SESM_Model(
    n_samples=n_samples,
    n_features=n_features,
    n_functions=l_functions,
    psi=gaussian_function.gaussian
    )

  model_epochs = m_epochs
  ista_epochs = h_epochs
  dictionary_epochs = dict_epochs

  ista_alpha = 0.06
  ista_lambd = 0.005

  dictionary_alpha = 0.06

  model.fit(
      X=X,
      y=y,
      model_epochs=model_epochs,
      ista_epochs=ista_epochs,
      ista_alpha=ista_alpha,
      ista_lambd=ista_lambd,
      dictionary_epochs=dictionary_epochs,
      dictionary_alpha=dictionary_alpha
      )


  x_tensor = torch.tensor(x_values)
  y_tensor = torch.tensor(y_values)
  XY = torch.cat((x_tensor.unsqueeze(1), y_tensor.unsqueeze(1)), dim=1)

  Z = model.predict(XY)

  time = model.time/60

  #Init MSE metric
  mse = MeanSquaredError()
  # Compute MSE
  mse(Z.clone().detach(), z_values)
  mse_value = mse.compute()


  if debug:
    fig = plt.figure(figsize=(17, 17))

    ax1 = fig.add_subplot(221, projection='3d')
    ax1.scatter(x_values, y_values, z_values,c=z_values)
    ax1.set_xlabel('X')
    ax1.set_ylabel('Y')
    ax1.set_zlabel('Z')
    ax1.set_title('Original Function')


    ax2 = fig.add_subplot(222, projection='3d')
    ax2.scatter(x_values, y_values, Z.clone().detach(), c=Z.clone().detach())
    ax2.set_xlabel('X')
    ax2.set_ylabel('Y')
    ax2.set_zlabel('Z')
    ax2.set_title('Surrogate Model')
    ax2.set_zlim(0)


    # Show the plot
    fname = f"figs_{expe}/{iter}.png"
    plt.savefig(fname)
    plt.show()

    plt.plot(model.losses)

    plt.ylabel('Loss')
    plt.xlabel('Iteration')

  return time, mse_value

In [47]:
import csv

def save_results(data, expe):
  # Calcular el promedio y la desviación estándar para time y mse
  times = [item[1] for item in data]
  mse_values = [item[2] for item in data]

  average_time = np.mean(times)
  std_time = np.std(times)
  average_mse = np.mean(mse_values)
  std_mse = np.std(mse_values)

  # Guardar los datos en un archivo CSV
  with open(f"resultados_{expe}.csv", mode="w", newline="") as file:
      writer = csv.writer(file)
      writer.writerow(["iter", "Tiempo (min)", "mse"])
      writer.writerows(data)
      writer.writerow(["Promedio", average_time, average_mse])
      writer.writerow(["Desviación Estándar", std_time, std_mse])


In [48]:
# 3 diag
xx, yy, zz = generate_mesh_and_z(sigma1_d, sigma2_d, sigma3_d)

# 2 diag, 1 no diag
xx_1, yy_1, zz_1 = generate_mesh_and_z(sigma1_d, sigma2_d, sigma3)

# 1 diag, 2 no diag
xx_2, yy_2, zz_2 = generate_mesh_and_z(sigma1_d, sigma2, sigma3)
# 3 no diag
xx_3, yy_3, zz_3 = generate_mesh_and_z(sigma1, sigma2, sigma3)

In [ ]:

fig = go.Figure(data=[go.Surface(z=zz_3.numpy(), x=xx_3, y=yy_3)])
fig.update_layout(scene=dict(aspectmode='data'))
fig.update_layout(scene=dict(camera=dict(eye=dict(x=2, y=2, z=1))))

fig.show()

In [ ]:
N_iter = 11
data = []
for i in range(N_iter):
  time, mse = run_experiment(xx,yy,zz,
                 m_epochs=25,
                 dict_epochs=800,
                 h_epochs=1000, expe=1, iter=i)
  data.append((i, time, mse.item()))

save_results(data=data, expe=1)

In [ ]:
from google.colab import files


files.download('resultados_1.csv')
!zip -r figs_1.zip figs_1/
files.download('figs_1.zip')

In [ ]:
data_1 = []

for i in range(N_iter):
  time, mse = run_experiment(xx_1,yy_1,zz_1,
                 m_epochs=25,
                 dict_epochs=800,
                 h_epochs=1000, expe=2, iter=i)
  data_1.append((i, time, mse.item()))

save_results(data=data_1, expe=2)

In [ ]:
files.download('resultados_2.csv')
!zip -r figs_2.zip figs_2/
files.download('figs_2.zip')

In [ ]:
data_2 = []
for i in range(N_iter):
  time, mse = run_experiment(xx_2,yy_2,zz_2,
                 m_epochs=25,
                 dict_epochs=800,
                 h_epochs=1000, expe=3, iter=i)
  data_2.append((i, time, mse.item()))

save_results(data=data_2, expe=3)

In [ ]:
files.download('resultados_3.csv')
!zip -r figs_3.zip figs_3/
files.download('figs_3.zip')

In [ ]:
data_3 = []
for i in range(N_iter):
  time, mse = run_experiment(xx_3,yy_3,zz_3,
                 m_epochs=25,
                 dict_epochs=800,
                 h_epochs=1000, expe=1, iter=i)
  data_3.append((i, time, mse.item()))

save_results(data=data_3, expe=3)

In [ ]:
files.download('resultados_4.csv')
!zip -r figs_4.zip figs_4/
files.download('figs_4.zip')